# Track accumulation: raw scene EDA\n\nFirst look at the raw RadarScenes structure before designing the multi-scan accumulation scheme (`track-accumulation` branch). Goal here is just to see what a single scene actually looks like: how `scenes.json` orders detections across the 4 sensors, and what one tracked object's points look like on sensor 2 (the sensor this whole project has used so far), before deciding how to merge several scans of the same track into one representation.\n\nNo modeling, no aggregation logic yet, purely inspection."

In [1]:
import json
import sys

import pandas as pd

sys.path.insert(0, "../scripts")
import dataloader as dl

SEQUENCE = "sequence_1"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## Scene ordering: one sensor per scene, round-robin

`scenes.json`'s keys are chronological timestamps, but each individual scene is a *single sensor's* measurement, not a synchronized 4-sensor frame. The very first scene of `sequence_1` (used in chat a moment ago) happened to be sensor 1, all static clutter, no tracked object."

In [2]:
seq_dir = dl.DATA_ROOT / SEQUENCE
with open(seq_dir / "scenes.json") as f:
    scene_keys = list(json.load(f)["scenes"].keys())

print(f"{SEQUENCE}: {len(scene_keys)} scenes total")
for k in scene_keys[:8]:
    det = dl.load_scene(SEQUENCE, int(k))
    print(k, "sensor_id:", sorted(set(det["sensor_id"].tolist())), "n_points:", len(det))

sequence_1: 4658 scenes total
156862647501 sensor_id: [1] n_points: 3
156862659751 sensor_id: [2] n_points: 97
156862695773 sensor_id: [4] n_points: 110
156862701077 sensor_id: [3] n_points: 101
156862719325 sensor_id: [1] n_points: 5
156862735209 sensor_id: [2] n_points: 92
156862768707 sensor_id: [4] n_points: 114
156862773081 sensor_id: [3] n_points: 104


## First sensor-2 scene with an actual tracked object

Scan scenes in chronological order, restrict each to sensor 2 (this project's sensor throughout), and stop at the first one that has at least one point with a non-empty `track_id` (a real labeled dynamic object, not static clutter)."

In [3]:
first_tracked_ts = None
for k in scene_keys:
    ts = int(k)
    det = dl.load_scene(SEQUENCE, ts)
    sensor2 = det[det["sensor_id"] == 2]
    if len(sensor2) == 0:
        continue
    if (sensor2["track_id"] != b"").any():
        first_tracked_ts = ts
        break

scene_index = scene_keys.index(str(first_tracked_ts))
print(f"Found at timestamp {first_tracked_ts} (scene #{scene_index} of {len(scene_keys)})")

Found at timestamp 156862659751 (scene #1 of 4658)


In [4]:
det = dl.load_scene(SEQUENCE, first_tracked_ts)
sensor2 = det[det["sensor_id"] == 2]

df = pd.DataFrame({
    "track_id": [tid.decode() or "(none/static)" for tid in sensor2["track_id"]],
    "label": [dl.LABELS[lid][0] for lid in sensor2["label_id"]],
    "x_cc": sensor2["x_cc"],
    "y_cc": sensor2["y_cc"],
    "range_sc": sensor2["range_sc"],
    "azimuth_sc": sensor2["azimuth_sc"],
    "rcs": sensor2["rcs"],
    "vr": sensor2["vr"],
    "vr_compensated": sensor2["vr_compensated"],
})
print(f"sensor 2, timestamp {first_tracked_ts}: {len(df)} points total")
df

sensor 2, timestamp 156862659751: 97 points total


,track_id,label,x_cc,y_cc,range_sc,azimuth_sc,rcs,vr,vr_compensated
0,(none/static),static,10.533686,5.161779,8.882484,1.156905,0.101503,-7.361293,0.244834
1,(none/static),static,11.381632,5.069020,9.479269,1.090472,-3.724234,-7.834501,0.193050
2,(none/static),static,11.521670,5.329981,9.749968,1.102969,-2.548794,-6.797734,1.153192
3,(none/static),static,11.780108,5.069591,9.798790,1.065767,-4.113401,-7.868203,0.307128
4,(none/static),static,12.161620,5.099507,10.126756,1.045971,-4.816975,-8.005029,0.285119
...,...,...,...,...,...,...,...,...,...
92,(none/static),static,59.482788,-21.690355,59.451572,0.075339,1.092561,-9.194952,0.206170
93,(none/static),static,23.307259,-2.011315,19.491419,0.368858,-20.475037,-9.915590,0.128433
94,(none/static),static,29.960392,-1.955132,26.130552,0.388134,-16.052677,-9.948433,0.107937
95,13ed213caa8511e89373ecf4bb1ae69c,car,41.911007,-0.174438,38.054634,0.449997,26.412235,-0.088121,9.982620


## The tracked instance itself

Isolate the non-empty `track_id` rows and add `x_rel`/`y_rel` (centroid-relative, `taxonomy_separability.add_relative_features`'s recipe), the same features the baseline MLP and DeepReflecs both consume. This single scan's version of the instance is the unit multi-scan accumulation will need to extend."

In [5]:
tracked = df[df["track_id"] != "(none/static)"].copy()
tracked["x_rel"] = tracked["x_cc"] - tracked["x_cc"].mean()
tracked["y_rel"] = tracked["y_cc"] - tracked["y_cc"].mean()

print(f"track_id: {tracked['track_id'].iloc[0]}, label: {tracked['label'].iloc[0]}, n_points: {len(tracked)}")
tracked[["track_id", "label", "x_rel", "y_rel", "range_sc", "rcs", "vr_compensated"]]

track_id: 13ed213caa8511e89373ecf4bb1ae69c, label: car, n_points: 2


,track_id,label,x_rel,y_rel,range_sc,rcs,vr_compensated
95,13ed213caa8511e89373ecf4bb1ae69c,car,-0.930347,0.121923,38.054634,26.412235,9.982620
96,13ed213caa8511e89373ecf4bb1ae69c,car,0.930344,-0.121923,39.912693,-7.088804,9.970025


## x_seq/y_seq: the same two points in the global frame

`x_cc`/`y_cc` is car-coordinate (ego-relative, shifts every scan). `x_seq`/`y_seq` is the dataset's own fixed global-sequence frame, odometry-corrected, per the RadarScenes Readme. This is the frame the whiteboard discussion landed on for cross-scan accumulation."

In [6]:
tracked["x_seq"] = sensor2["x_seq"][tracked.index]
tracked["y_seq"] = sensor2["y_seq"][tracked.index]
tracked[["track_id", "label", "x_cc", "y_cc", "x_seq", "y_seq", "range_sc", "rcs", "vr_compensated"]]

,track_id,label,x_cc,y_cc,x_seq,y_seq,range_sc,rcs,vr_compensated
95,13ed213caa8511e89373ecf4bb1ae69c,car,41.911007,-0.174438,-204.212921,-248.272385,38.054634,26.412235,9.982620
96,13ed213caa8511e89373ecf4bb1ae69c,car,43.771698,-0.418284,-205.954453,-248.971451,39.912693,-7.088804,9.970025


## The whole track: every sensor-2 detection of this track_id in the sequence

Scan every scene in `sequence_1`, keep sensor-2 points matching this exact `track_id`, across its full lifetime (not just the one scene above)."

In [7]:
target_track = tracked["track_id"].iloc[0].encode()
rows = []
for k in scene_keys:
    ts = int(k)
    det = dl.load_scene(SEQUENCE, ts)
    sensor2 = det[det["sensor_id"] == 2]
    hits = sensor2[sensor2["track_id"] == target_track]
    for row in hits:
        rows.append({
            "timestamp": ts,
            "x_cc": row["x_cc"], "y_cc": row["y_cc"],
            "x_seq": row["x_seq"], "y_seq": row["y_seq"],
            "range_sc": row["range_sc"], "rcs": row["rcs"], "vr_compensated": row["vr_compensated"],
        })

track_df = pd.DataFrame(rows)
print(f"track {target_track.decode()}: {len(track_df)} points across {track_df['timestamp'].nunique()} scenes")
track_df

track 13ed213caa8511e89373ecf4bb1ae69c: 2882 points across 830 scenes


,timestamp,x_cc,y_cc,x_seq,y_seq,range_sc,rcs,vr_compensated
0,156862659751,41.911007,-0.174438,-204.212921,-248.272385,38.054634,26.412235,9.982620
1,156862659751,43.771698,-0.418284,-205.954453,-248.971451,39.912693,-7.088804,9.970025
2,156862735209,41.943661,-0.169863,-204.885651,-248.702972,38.087349,25.881981,9.979215
3,156862735209,43.653202,-0.272108,-206.425156,-249.453217,39.795502,-8.690852,9.970519
4,156862808076,41.930992,-0.042363,-205.435928,-249.211487,38.076672,24.283150,9.974039
...,...,...,...,...,...,...,...,...
2877,156924398108,7.835287,0.080014,-706.155396,-273.683990,4.051089,-7.479878,-0.020208
2878,156924398108,9.279313,0.789984,-707.587036,-274.418518,5.620409,-8.461930,-0.012597
2879,156924398108,9.582397,0.877265,-707.888611,-274.510986,5.935790,-13.759801,-0.435510
2880,156924398108,10.332762,0.904260,-708.638367,-274.550781,6.668605,-8.585849,-0.005246


## Does this confirm the motion-smearing concern from the whiteboard discussion?

Compare the track's actual displacement in the fixed global frame (`x_seq`/`y_seq`, first scene to last) against what `vr_compensated` alone would predict, as a sanity check that the two agree, and to put a real number on how far this object moves over its own tracked lifetime."

In [8]:
first, last = track_df.iloc[0], track_df.iloc[-1]
duration_s = (last["timestamp"] - first["timestamp"]) / 1e6
displacement = ((last["x_seq"] - first["x_seq"]) ** 2 + (last["y_seq"] - first["y_seq"]) ** 2) ** 0.5

print(f"lifetime: {duration_s:.2f} s across {track_df['timestamp'].nunique()} scenes")
print(f"x_seq/y_seq displacement, first to last: {displacement:.2f} m")
print(f"implied average speed: {displacement / duration_s:.2f} m/s")
print(f"vr_compensated, mean over track: {track_df['vr_compensated'].mean():.2f} m/s")
print()
print("A single accumulated point set built from every one of these scenes, without any")
print("per-scan recentering on the object's own position, would span this whole displacement,")
print(f"~{displacement:.0f} m, far larger than the car's own physical extent (the two points in")
print("one scene above sit under 2 m apart). This is the smearing risk from the whiteboard chapter,")
print("now with a concrete number attached instead of a hypothetical.")

lifetime: 61.74 s across 830 scenes
x_seq/y_seq displacement, first to last: 501.63 m
implied average speed: 8.13 m/s
vr_compensated, mean over track: 7.32 m/s

A single accumulated point set built from every one of these scenes, without any
per-scan recentering on the object's own position, would span this whole displacement,
~502 m, far larger than the car's own physical extent (the two points in
one scene above sit under 2 m apart). This is the smearing risk from the whiteboard chapter,
now with a concrete number attached instead of a hypothetical.
